# Dates


# 4 July 2025 




In [ ]:
import easyocr
import cv2
import re
from datetime import datetime
import numpy as np
import os
import gc
import csv

In [ ]:
# Regex patterns
datepatn = r'\d{2}[-/]\d{2}[-/]\d{4}'
panpatn = r'([A-Z]){5}([0-9]){4}([A-Z]){1}'

def needs_cropping(img):
    keywords = [
        'please inform', 'return to',
        'pan services unit', 'protean', 'tin', 'nsdl',
        'sapphire chambers', 'baner road', 'pune',
        '@', 'tel:', 'email'
    ]

    ocr_lines = reader.readtext(img, detail=0, width_ths=0.9)
    full_text = ' '.join(ocr_lines).lower()

    for key in keywords:
        if key in full_text:
            print(f"🔍 Detected back-side indicator: '{key}' → will crop")
            return True

    print("✅ Front-only PAN detected — no cropping needed.")
    return False



def crop_front_from_combined(img, save_path=None):
    results = reader.readtext(img)

    # Find x-coordinate for "INCOME TAX" or "GOVT OF INDIA"
    left_x = None
    for (bbox, text, prob) in results:
        line = text.lower()
        if 'income tax' in line or 'govt of india' in line:
            # Get left-most x from bounding box
            left_x = min([pt[0] for pt in bbox])
            break

    h, w, _ = img.shape

    if left_x is None:
        print("⚠️ Could not locate front-side anchor, using full image.")
        cropped = img
    else:
        crop_left = max(0, int(left_x) - 30)
        crop_right = min(w, crop_left + int(0.5 * w))  # Crop 50% width from anchor
        cropped = img[:, crop_left:crop_right]

    # Save cropped image
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, cropped)
        print(f"🖼️ Cropped image saved to: {save_path}")

    return cropped




In [ ]:
import re
from difflib import get_close_matches

# List of fuzzy PAN-related label phrases
PAN_LABELS = [
    "permanent account number",
    "Permanent Acccunt Number",
    "Permanent Account Num",
    "Permanent Account Number",
    "Pemancnt Account Numner",
    "Pemancnt Account Numner",
    "permanent account number card",
    "Permanent Accaunt Number Card",
    "perianenl accaunt number card",
    "Permanent Account Number Card",
    "Permanent Accaunt Number Card",
    "pernanent account nuinbor gard",
    "Potmtgnt Account Muner Cadl",
    "Permanert Account Humber Card",
    "Permanent Account Number Card",
    "Parinanenl Accouni Hunbys",
    "pc:manent account number",
    "p=ra accou nunter cad",
    "Pertaaner Acceusi iumuei C#g",
    "maneninccouni Number",
    "Peinnncnl nec, 'Number"
    "Permanent Account Nu uar"
    "permanent account nu",
    "Permanent Account Nu",
    "Pennanent Account N",
    "Permanent Account Ni",
    "Permanent Accounl",
    "PPERIAANE#T",
    "pan number",
    "account number",
    "account mumost",
    "account number card",
    "Acrnunt Munner Cur",
    "acrnunt munner cur",
    "permanent account number (e-pan) card",
    "e. permanent account number cyc",
    "e. Pemanent Accounl Nunber Cyc",
    "Feranei Accoun Vunber Card",
    "Ferinanent Accnunt humger Card"
    "Feranei Accoun Vunber Card"
    , "Fermarer! AceLur* HuLE+"
    , "Permaneni Accouni Numoer"
    , "Permanent Account Nun"
    , "Perancnt Accqunt Number Card"
    , "Perranent Account Number"
    , "Permanen! Account Number Card"
    , "Pelmanent Account Mumber"
    , "Perranent Account Number"
    , "Peirsnt ccount Numeer Card"
    , "Fermanent Accoulli Namber"
    , "Permanent nccgunt Nuber Card"
    , "Punmnent Avvenut Nurnlet"
    , "Peransnt Accounl Number Cad"
    , "Pernanent Account Numnber Card"
    , "Permanent Accaunt Number Caru"
    , "Permenlart Account Numiber"
    , "Permnanent Account Nulnber Card"
    , "Penaner   Number Card"
    , "Pormanent Account Mumber"
    , "ferntltlmcLuunl fuu TCerLalD"
    , "Permanent Account Mumber Card"
    , "Permanent Account Humber Card"
    
    
]

import re

def is_pan_label(line, debug=False):
    """
    Check if a line is a fuzzy match to PAN-related label phrases.
    """
    normalized = re.sub(r'[^a-z]', '', line.lower())
    for label in PAN_LABELS:
        norm_label = re.sub(r'[^a-z]', '', label.lower())
        if norm_label in normalized:
            # if debug: print(f"✅ Matched PAN label: {line}")
            return True
    return False

def clean_pan_token(token):
    token = token.upper()
    token = re.sub(r'[^A-Z0-9]', '', token)
    
    # Replace common OCR confusion ONLY in digit positions
    if len(token) == 10:
        token = (
            token[:5] +
            token[5:9].replace('O', '0').replace('I', '1') +  # digit part
            token[9:]
        )
    return token

def extract_pan_from_ocr_lines(ocr_lines, debug=False):
    pan_pattern = re.compile(r'^[A-Z]{5}[0-9]{4}[A-Z]$')

    for i, line in enumerate(ocr_lines):
        if is_pan_label(line, debug):
            #if debug: print(f"🔍 Found PAN label: {line}")
            for offset in range(1, 3):  # next 2 lines
                if i + offset < len(ocr_lines):
                    next_line = ocr_lines[i + offset]
                    tokens = next_line.split()
                    #if debug: print(f"➡️ Checking line: {next_line}")
                    for token in tokens or [next_line]:  # handle full line too
                        cleaned = clean_pan_token(token)
                        #if debug: print(f"🧪 Checking token: {token} -> {cleaned}")
                        if len(cleaned) == 10 and pan_pattern.fullmatch(cleaned):
                            #if debug: print(f"✅ Matched PAN: {cleaned}")
                            return cleaned
    return 'NAN'




In [ ]:
def normalize_ocr_text(text):
    fixed = (text.replace('|', '/')
                 .replace('I', '1')
                 .replace('l', '1')
                 .replace('O', '0'))
    return re.sub(r'(\d{2})[^\d](\d{2})[^\d](\d{4})', r'\1/\2/\3', fixed)


def is_probably_person_name(text):
    text = text.strip()
    if not text or len(text) < 5:
        return False
    if re.fullmatch(r'[A-Z]{5}[0-9]{4}[A-Z]', text):  # Avoid PANs
        return False
    if any(keyword in text.lower() for keyword in ['account', 'number', 'signature', 'govt', 'income']):
        return False
    if all(c.isalpha() or c.isspace() for c in text):
        return True
    return False

def extract_name_and_father_name1(ocr_lines):
    name, father_name = 'NAN', 'NAN'

    for i, line in enumerate(ocr_lines):
        line_clean = line.lower().strip()

        # Improved Name detection
        if re.search(r'(name|mame|kame|titi.?name|=ih.?name|namlle|namle)', line_clean) and 'father' not in line_clean:
            if i + 1 < len(ocr_lines):
                candidate = ocr_lines[i + 1].strip()
                if is_probably_person_name(candidate):
                    name = candidate
                    continue

        # Father name fuzzy match
        if 'father' in line_clean and re.search(r'(name|nanie|namie)', line_clean):
            if i + 1 < len(ocr_lines):
                candidate = ocr_lines[i + 1].strip()
                if is_probably_person_name(candidate):
                    father_name = candidate

    return name, father_name

def read_ocr_text_from_image(img):
    """
    Runs EasyOCR on the image and returns list of text lines.
    """
    OCR_text = reader.readtext(img, detail=0, width_ths=0.9)
    ocr_lines = [txt.strip() for txt in OCR_text if txt.strip()]
    return ocr_lines

def extract_pan_details_from_lines(ocr_lines, debug=False):
    """
    Extracts PAN number, Name, Father's Name, and DOB from OCR lines.
    """
    datepatn = r'\b\d{1,2}[-/]\d{1,2}[-/]\d{4}\b'

    # Init
    PAN = 'NAN'
    Name = 'NAN'
    FatherName = 'NAN'
    DOB = 'NAN'

    # PAN
    PAN = extract_pan_from_ocr_lines(ocr_lines, debug)

    # DOB
    for text in ocr_lines:
        norm_text = normalize_ocr_text(text)
        match = re.search(datepatn, norm_text)
        if match:
            DOB = match.group()
            break

    # Try to extract Name and Father's Name by keyword
    Name, FatherName = extract_name_and_father_name1(ocr_lines)
    # if debug: print("Name (after fuzzy):", Name)
    # if debug: print("FatherName (after fuzzy):", FatherName)

    # Fallback if Name or FatherName is still NAN
    if Name == 'NAN' or FatherName == 'NAN':
        gov_index = -1
        for i, line in enumerate(ocr_lines):
            if 'govt' in line.lower() or 'income' in line.lower():
                gov_index = i
                break

        if gov_index != -1:
            for j in range(gov_index + 1, len(ocr_lines)):
                candidate = ocr_lines[j].strip()
                if is_probably_person_name(candidate):
                    if Name == 'NAN':
                        Name = candidate
                    elif FatherName == 'NAN':
                        FatherName = candidate
                    if Name != 'NAN' and FatherName != 'NAN':
                        break

    extracted = {
        'Pan_number': PAN,
        'Name': Name,
        'Father_Name': FatherName,
        'DOB': DOB
    }

    return extracted


def is_all_nan(extracted):
    return all(value == 'NAN' for value in extracted.values())


# execute code against images

In [ ]:
import easyocr
import cv2
import re
from datetime import datetime
import os
import gc
import json
import csv

# Initialize EasyOCR reader once (CPU only)
reader = easyocr.Reader(['en'], gpu=False)

# Directory containing PAN card images
input_dir = './PAN/testscenarios/worked/all/allimages/check'
output_crop_dir = os.path.join(input_dir, 'cropped')

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Directory where the CSV will be saved
input_dir = './PAN/testscenarios/worked/all/allimages/check'

# Output path with timestamp
output_csv_path = os.path.join(input_dir, f'pan_extraction_log_{timestamp}.csv')

os.makedirs(output_crop_dir, exist_ok=True)

# Extensions to look for
valid_exts = ['.jpg', '.jpeg', '.png']

# 🔁 Get all image files
file_list = [os.path.join(input_dir, f) for f in os.listdir(input_dir)
             if os.path.splitext(f)[1].lower() in valid_exts]

print(f"\n🔍 Found {len(file_list)} PAN images to process.\n")

# Open CSV writer
with open(output_csv_path, mode='w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image_file', 'ocr_lines', 'Pan_number', 'Name', 'Father_Name', 'DOB']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for image_file in sorted(file_list):
        print(f"📄 Processing: {os.path.basename(image_file)}")
        start_time = datetime.now()
        print(f"🔹 Start Time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

        original_img = cv2.imread(image_file)
        if original_img is None:
            print("⚠️ Could not read image. Skipping.")
            continue

        # Step 1: Crop front from combined if needed
        cropped_path = os.path.join(output_crop_dir, f"cropped_{os.path.basename(image_file)}")
        if needs_cropping(original_img):
            original_img = crop_front_from_combined(original_img, save_path=cropped_path)

        angles = [0, 90, 180, 270]
        extracted = None
        ocr_lines = []

        for angle in angles:
            if angle == 90:
                img = cv2.rotate(original_img, cv2.ROTATE_90_CLOCKWISE)
            elif angle == 180:
                img = cv2.rotate(original_img, cv2.ROTATE_180)
            elif angle == 270:
                img = cv2.rotate(original_img, cv2.ROTATE_90_COUNTERCLOCKWISE)
            else:
                img = original_img.copy()

            # OCR & extraction
            #extracted, ocr_lines = extract_pan_details(img)
            ocr_lines = read_ocr_text_from_image(img)  # step 1
            extracted = extract_pan_details_from_lines(ocr_lines)  # step 2

            del img
            gc.collect()

            if not is_all_nan(extracted):
                print(f"✅ Success with {angle}° rotation")
                break
            else:
                print(f"❌ All NAN with {angle}° rotation. Trying next...")

        # Final cleanup
        del original_img
        gc.collect()

        end_time = datetime.now()
        duration = end_time - start_time

        print("📂 File:", os.path.basename(image_file))
        print("🧾 OCR Text:", ocr_lines)
        print("✅ Extracted:", extracted)
        print(f"⏱ Time Taken: {duration}")
        print("-" * 80)

        # Save to CSV using JSON for ocr_lines
        writer.writerow({
            'image_file': os.path.basename(image_file),
            'ocr_lines': json.dumps(ocr_lines, ensure_ascii=False),
            'Pan_number': extracted.get('Pan_number', 'NAN'),
            'Name': extracted.get('Name', 'NAN'),
            'Father_Name': extracted.get('Father_Name', 'NAN'),
            'DOB': extracted.get('DOB', 'NAN')
        })

print(f"\n✅ All done. Output saved to: {output_csv_path}")


# Run regression checks


In [ ]:
import pandas as pd
import glob
import json
import time

# === 🧩 Configurable Options ===
target_columns = ['Pan_number']                      # Check specific columns or all
filter_on_nan_column = 'Pan_number'                  # Only rows with NAN in this column
run_extraction = True                                # Toggle re-running extraction
debug = False

# === 🔍 Step 1: Load and prepare data ===
csv_files = glob.glob('./PAN/testscenarios/worked/pan_extraction_log_*.csv')
#csv_files = glob.glob('./PAN/testscenarios/worked/singletest_*.csv')
df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df_all['ocr_lines'] = df_all['ocr_lines'].apply(json.loads)

# === 📌 Optional Filter on NAN column ===
if filter_on_nan_column:
    df_all = df_all[df_all[filter_on_nan_column] == 'NAN']
    print(f"🔎 Filtered on NAN in '{filter_on_nan_column}': {len(df_all)} rows to recheck")

    # 📝 Save test scenario file
    timestamp = time.strftime("%Y%m%d%H%M%S")
    test_scenario_path = f'./PAN/testscenarios/test_scenario_{filter_on_nan_column}_NAN_{timestamp}.csv'
    df_all.to_csv(test_scenario_path, index=False)
    print(f"📝 Saved filtered test scenario to: {test_scenario_path}")

# === 🧠 Function to Re-run Extraction ===
def re_extract(row):
    if run_extraction:
        if debug: print(f"\n🧾 OCR Text:", row['ocr_lines'])
        re_extracted = extract_pan_details_from_lines(row['ocr_lines'], True)
        if debug: print(f"✅ Extracted:", re_extracted)
        return pd.Series({
            'Pan_number_new': re_extracted['Pan_number'],
            'Name_new': re_extracted['Name'],
            'Father_Name_new': re_extracted['Father_Name'],
            'DOB_new': re_extracted['DOB']
        })
    else:
        return pd.Series({
            'Pan_number_new': row.get('Pan_number_new', ''),
            'Name_new': row.get('Name_new', ''),
            'Father_Name_new': row.get('Father_Name_new', ''),
            'DOB_new': row.get('DOB_new', '')
        })

# === 🔁 Apply extraction
if run_extraction:
    df_all[['Pan_number_new', 'Name_new', 'Father_Name_new', 'DOB_new']] = df_all.apply(re_extract, axis=1)

# === ✅ Compare selected columns
for col in target_columns:
    df_all[f'{col}_match'] = df_all[col] == df_all[f'{col}_new']

# === 🚩 Identify mismatches
mismatch_condition = ~df_all[[f"{col}_match" for col in target_columns]].all(axis=1)
mismatches = df_all[mismatch_condition]

print(f"\n🧪 Regression completed on columns {target_columns}: {len(mismatches)} mismatches out of {len(df_all)} checked.\n")

# === 🖨️ Show mismatches
print_cols = ['image_file'] + sum([[col, f"{col}_new"] for col in target_columns], [])
print(mismatches[print_cols])

# === 💾 Save mismatches to timestamped file
if not mismatches.empty:
    mismatch_ts = time.strftime("%Y%m%d%H%M%S")
    mismatch_path = f'./PAN/testscenarios/mismatches_after_regression_{mismatch_ts}.csv'
    mismatches.to_csv(mismatch_path, index=False)
    print(f"📁 Mismatches saved to: {mismatch_path}")
else:
    print("✅ No mismatches found after regression.")


📄 Processing: C1119_Pan_133693887196035159 - Copy.jpg
🔹 Start Time: 2025-07-10 14:32:28
🔍 Detected back-side indicator: '@' → will crop
⚠️ Could not locate front-side anchor, using full image.
🖼️ Cropped image saved to: ./PAN/testscenarios/worked/all/allimages/check\cropped\cropped_C1119_Pan_133693887196035159 - Copy.jpg
✅ Success with 0° rotation
📂 File: C1119_Pan_133693887196035159 - Copy.jpg
🧾 OCR Text: ['3ITZTIK feT', 'HIG7rE', 'INCOMETAXDEPARTMENT', 'GOVI. OEILLA', 'DASARI RAIA RAO', 'PYDITALLY DASARI', '01/07/1970', 'Permanent Acccunt Number', 'BDNPD8867K', '&(me@;', 'slnnature']
✅ Extracted: {'Pan_number': 'BDNPD8867K', 'Name': 'DASARI RAIA RAO', 'Father_Name': 'PYDITALLY DASARI', 'DOB': '01/07/1970'}
⏱ Time Taken: 0:00:35.343959

-----------------------------



In [ ]:
tensorflow==2.11
keras==2.11
numpy==1.24.4
pillow==9.5.0
easyocr
keras-ocr
pandas==1.5.3


pip install -r requirements.txt

In [ ]:
import pandas as pd
import os
import shutil

# === CONFIGURATION ===
csv_path = './PAN/testscenarios/worked/ocrissuemove01.csv'  # Path to your CSV file
source_folder = './PAN/testscenarios/worked/toprocess-1'  # Folder where the original images are stored
destination_folder = './PAN/testscenarios/worked/ocr-reading-issue'  # Folder to move files to

# === STEP 1: Read the CSV ===
df = pd.read_csv(csv_path)

# === STEP 2: Ensure destination folder exists ===
os.makedirs(destination_folder, exist_ok=True)

# === STEP 3: Move files listed in the image_file column ===
for filename in df['image_file']:
    src_path = os.path.join(source_folder, filename)
    dst_path = os.path.join(destination_folder, filename)

    if os.path.exists(src_path):
        shutil.move(src_path, dst_path)
        print(f"Moved: {filename}")
    else:
        print(f"File not found: {filename}")
